# Installing Python, IDEs, and the REPL

**Topic:** `1.2` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Turn 'it works on my machine' into a fact you can print. This topic covers how an installation is actually put together, how to identify the running interpreter, and how to build a pre-flight check any deployment can call.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

Most Python problems that are not logic problems are environment problems. Knowing exactly which interpreter runs, which packages it can see, and how to prove both in one command is the difference between a five-minute fix and an afternoon of guessing.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Identify the running interpreter and its version programmatically.
2. Explain how PATH and sys.path decide which program and module wins.
3. Create and inspect a virtual environment using the standard library.
4. Detect an optional dependency without importing it.
5. Build a pre-flight readiness check that returns an exit status.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.1 Introduction to Python
- Ability to open a terminal and run a command.

## 4. Mental Model

**Mental model: an installation is a stack of three rings.** The innermost ring is the *interpreter binary*. Around it sits an *environment* that owns its own package directory. Outside both is the *PATH*, which decides which of many possible binaries answers when you type a name. A command reaches your code only if it passes all three rings, and it can be shadowed at any one of them.

| If this is wrong | The symptom you will see |
| --- | --- |
| Interpreter ring | Wrong version reported, or a syntax error on valid code |
| Environment ring | 'ModuleNotFoundError' for a library you just installed |
| PATH ring | `python3` runs, `python` is missing or is a different version |
| sys.path order | A local file shadows a standard library module |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Interpreter | The program that reads and executes Python source code. |
| Launcher | The command such as `python3` that finds and starts an interpreter. |
| PATH | Ordered list of directories searched for an executable by name. |
| Virtual environment (venv) | A directory holding its own package tree and config. |
| site-packages | The directory inside an environment where packages are installed. |
| sys.path | The ordered list of directories searched for importable modules. |
| sys.executable | Absolute path of the interpreter currently running your code. |
| sys.prefix | Installation prefix of the active environment. |
| REPL | Read-Eval-Print Loop: the interactive prompt. |
| find_spec | Locate a module without importing it, used for optional dependencies. |
| PEP 405 | The specification that defines Python virtual environments. |

## 6. Conceptual Explanation

Installing Python is not one action but three separate decisions, and most "Python does not work" problems are really arguments between them. The first decision is **which interpreter**: the system package, a version manager, a container, or a bundled copy inside an IDE. The second is **which environment**: the base installation or an isolated virtual environment that owns its own site-packages directory. The third is **which launcher**: on many systems `python` and `python3` are different executables, and only one of them is yours.

These three decisions are not cosmetic. If your editor runs the system interpreter while your terminal activates a virtual environment, your tests will pass locally and fail on the build machine, or your newly installed package will appear to vanish. Every experienced engineer has lost an hour to this, so the goal of this topic is to make the interpreter *identifiable* rather than assumed.

Python is a cross-platform language, which means the same source runs on Linux, macOS and Windows, but the surrounding tooling is not identical. On Linux and macOS the conventional launcher is `python3`; on Windows the conventional launcher is `python`, and the official installer additionally offers `py`. The language itself is identical in all cases; only the paths, separators and launcher names differ. Anything you write that must run everywhere should therefore avoid hard-coded separators and prefer the `pathlib` module introduced in Module 7.

In [ ]:
import sys
import platform

print('executable :', sys.executable)
print('version    :', sys.version.split()[0])
print('platform   :', platform.system(), platform.machine())
print('max unicode:', sys.maxunicode)

In [ ]:
import sys

print('interactive (REPL):', hasattr(sys, 'ps1'))
print('frozen (packaged) :', getattr(sys, 'frozen', False))
print('prefix            :', sys.prefix)
print('base_prefix       :', sys.base_prefix)

> **Why base_prefix exists**
>
> In a standard virtual environment `sys.prefix` equals `sys.base_prefix`, because the environment shares the base installation's interpreter. Inside a venv the two values differ in the opposite direction on some platforms, and an activated venv normally does *not* change `sys.executable` at all - which is exactly why checking it in every script is so useful.

## 7. Formal Theory

When you type a command, the operating system resolves the name to a file using the **PATH** environment variable, an ordered list of directories. The first matching executable wins. This single rule explains an enormous number of confusing behaviours: a second Python installed later in the PATH silently shadows an earlier one, and `pip install` may target a different interpreter than the `python` you just verified.

```text
command name  ->  scan $PATH left to right  ->  first executable match  ->  run
```

Inside the interpreter, a parallel mechanism governs imports. `sys.path` is an ordered list of directories searched for modules; the current directory is typically first when you run a script, which is why a local file named `random.py` can shadow the standard library's `random` module. Module search order, not the filesystem, decides what `import random` actually loads.

In [ ]:
import sys
import shutil

print('which python :', shutil.which('python3'))
print('sys.path[0]  :', sys.path[0])
print('entries      :', len(sys.path))
for index, entry in enumerate(sys.path[:4]):
    print(f'  [{index}] {entry}')

| Concept | What it controls | Symptom when wrong |
| --- | --- | --- |
| Interpreter | Which Python binary runs | Version mismatch errors |
| Environment (venv) | Which packages are visible | 'Module not found' for an installed library |
| PATH order | Which launcher wins | Wrong version silently used |
| sys.path | Which module wins an import | Your file shadows a stdlib module |

## 8. Syntax

There is no special syntax for environments, but there are two idioms you will use constantly: comparing version tuples, and asking the interpreter a question at runtime.

In [ ]:
import sys

REQUIRED = (3, 12)          # this course requires 3.12 or newer
running = tuple(sys.version_info[:2])
if running >= REQUIRED:
    print(f'OK: {running} satisfies {REQUIRED}')
else:
    print(f'TOO OLD: {running} does not satisfy {REQUIRED}')

Note that `sys.version_info` compares correctly as a tuple, so `>=` needs no string comparison. Comparing `str(sys.version_info[:2])` lexicographically is a real bug: `'3.9'` sorts above `'3.12'` because `9` follows `1`.

> **Never compare versions as strings**
>
> '3.10' < '3.9' is True for strings and False for tuples. Always compare `sys.version_info[:2]` as a tuple, or use `packaging.version` for real dependency specifications.

## 9. Basic Examples

**Example 1 - confirm which interpreter is running.**

In [ ]:
import sys

print('running:', sys.executable)
print('version:', '.'.join(map(str, sys.version_info[:3])))

**Example 2 - inspect where packages come from.**

In [ ]:
import site

for path in site.getsitepackages():
    print('site-packages:', path)

## 10. Intermediate Examples

An environment manager is just a program that creates a directory and points an interpreter at it. You can drive the standard library's `venv` module from Python, which is exactly what an IDE does when it creates an environment for you.

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

target = Path(tempfile.gettempdir()) / 'demo_env'
result = subprocess.run(
    [sys.executable, '-m', 'venv', str(target)],
    capture_output=True,
    text=True,
)
print('returncode:', result.returncode)
print('created   :', (target / 'bin').exists() or (target / 'Scripts').exists())

> **Standard library only**
>
> Everything in this topic uses the standard library, so it works before any third-party package is installed. That is deliberate: environment problems are exactly when you cannot rely on your packages.

## 11. Advanced Examples

When a module cannot be imported, the useful question is not "is it installed?" but "would this import resolve, and from where?". `importlib.util.find_spec` answers the first without executing the module, which is the safe way to make optional-dependency decisions.

In [ ]:
import importlib.util

for name in ('json', 'requests', 'definitely_not_installed'):
    spec = importlib.util.find_spec(name)
    origin = spec.origin if spec else None
    print(f'{name:<26} -> {origin}')

Note that `find_spec` can return a value for a module that will still fail to import, because a parent package may raise during its own initialisation. It answers "where would this be loaded from", which is a question about search order rather than about runtime health.

## 12. Code Walkthrough

The programme below is a real pre-flight check. It is the kind of script a robotics team runs on every machine before arming hardware, and it is small enough to read in one screen and strict enough to be useful.

In [ ]:
"""verify_env.py - report whether this machine can run ROBO-X."""
from __future__ import annotations

import importlib.util
import shutil
import sys

REQUIRED_VERSION = (3, 12)
REQUIRED_MODULES = ('json', 'sqlite3', 'unittest.mock')


def check_version() -> tuple[bool, str]:
    running = tuple(sys.version_info[:2])
    return running >= REQUIRED_VERSION, '.'.join(map(str, running))


def check_modules() -> tuple[list[str], list[str]]:
    found = [name for name in REQUIRED_MODULES
             if importlib.util.find_spec(name) is not None]
    missing = [name for name in REQUIRED_MODULES if name not in found]
    return found, missing


def main() -> int:
    ok_version, version = check_version()
    found, missing = check_modules()
    print(f'interpreter : {sys.executable}')
    print(f'version     : {version} ({"ok" if ok_version else "too old"})')
    print(f'modules     : {len(found)}/{len(REQUIRED_MODULES)} present')
    for name in missing:
        print(f'  MISSING {name}')
    ready = ok_version and not missing
    print('READY' if ready else 'NOT READY')
    return 0 if ready else 1


if __name__ == '__main__':
    main()

> **Why not raise SystemExit here?**
>
> A real script ends with `raise SystemExit(main())` so the shell receives an exit status. Running that inside a notebook would terminate the kernel, so the cell calls `main()` directly. Keep the SystemExit version for the command-line tools you build later in the course.

The important property is that `main()` returns a status code rather than exiting from inside a helper. A shell script can therefore use the exit code to decide whether to arm the robot, and a test can assert on the same integer without spawning a process.

## 13. Line-by-Line Explanation

1. `REQUIRED_VERSION` and `REQUIRED_MODULES` are module-level constants: the policy lives in exactly one place and is easy to audit or change.
2. `check_version()` compares tuples, never strings, and returns both a verdict and the version string so the caller can report it.
3. `check_modules()` uses `find_spec`, which inspects the search path without importing, so a missing optional dependency cannot crash the check.
4. A list comprehension keeps the found/missing computation to one readable pass each.
5. `main()` prints a human-readable report and reduces the two checks to one boolean named `ready`.
6. The `raise SystemExit(main())` guard at the bottom turns that boolean into a process exit status, while keeping `main()` importable and testable.

## 14. Common Mistakes

**Mistake 1 - installing into the base interpreter and expecting a venv to see it.** A virtual environment created with `--system-site-packages` does inherit the base packages; the default, isolated one does not.

```text
# wrong: pip may target a different interpreter than the one you tested
$ pip install requests
$ python -c "import requests"    # ModuleNotFoundError

# right: bind the installer to the interpreter explicitly
$ python -m pip install requests
$ python -c "import requests"    # succeeds
```

**Mistake 2 - assuming `python` and `python3` are the same executable.**

```text
# wrong: bare 'python' may be absent, or may be Python 2 on old systems
python --version

# right: ask the interpreter you are actually going to use
python3 --version
python3 -c "import sys; print(sys.executable)"
```

**Mistake 3 - shadowing a standard library module with a local file.**

```text
# a file called random.py in the working directory
import random          # imports YOUR file, not the standard library
print(random.__file__) # reveals the truth immediately
```

**Mistake 4 - comparing versions as strings.**

```text
# wrong
if '3.9' > '3.12':      # True - string comparison

# right
if (3, 9) > (3, 12):    # False - tuple comparison
```

## 15. Debugging Techniques

Environment problems are diagnosed by *observation*, not by guessing. Three commands answer almost every question.

1. **Who am I?** `python -c "import sys; print(sys.executable)"` identifies the interpreter.
2. **What can I see?** `python -c "import sys; print(*sys.path, sep='\n')"` shows the module search order.
3. **What got installed?** `python -m pip list` shows packages for *that* interpreter, not for whatever `pip` happens to point at.

In [ ]:
import sys

print('first three search paths:')
for entry in sys.path[:3]:
    print('   ', entry)

print('current dir first?', sys.path and sys.path[0] in ('', '.', None))

> **Read the REPL prompt**
>
> A prompt containing `(...venv)` means a virtual environment is active. Python exposes this programmatically as `hasattr(sys, 'ps1')`, which is a reliable way for a script to refuse to run inside the wrong environment.

## 16. Best Practices

- Pin the interpreter, not just the package: record `python_requires` in `pyproject.toml`.
- Always install with `python -m pip`, never bare `pip`, so the installer and the interpreter cannot disagree.
- Keep one virtual environment per project, created from a committed requirements file.
- Use `shutil.which('name')` to resolve a launcher instead of guessing a path.
- Avoid hard-coded separators such as `/` or `\\`; let `pathlib` build paths.
- Put environment checks in a script that returns an exit code, so CI can call it too.

## 17. Performance Considerations

Import time is the one performance concern that belongs in an introductory topic, because a slow start is felt on every single run of a robotics service. Importing a large library such as `pandas` costs hundreds of milliseconds before your first line of logic executes. Two rules follow. First, import at module level rather than inside a function: a function-local import re-runs the search on every call and defeats the module cache. Second, delay genuinely optional imports until the feature that needs them is actually used, so a program that never reads a CSV never pays for the dataframe stack.

The trade-off is readability. A single top-level `import pandas` at the top of a notebook is clearer than a lazy loader, and for analysis code the cost is irrelevant. It is the small, frequently started command-line tools - the ones a robot launches on every mission - where import discipline genuinely changes what the user experiences.

> **Measure with the flag**
>
> `python -X importtime script.py` prints the time spent importing every module, largest first. It is the fastest way to find an unexpected heavy dependency.

## 18. Pythonic Approaches

Building the same path two ways:

In [ ]:
import os
from pathlib import Path

home = os.path.expanduser('~')

# string concatenation with a hard-coded separator
old = home + '/' + 'venvs' + '/' + 'robo'

# pathlib composes and normalises the same path
new = Path(home) / 'venvs' / 'robo'

print('old:', old)
print('new:', new)
print('equal:', Path(old) == new)

## 19. Robotics Connection

Every robot in a fleet runs on a machine whose environment someone verified. The simulator is deterministic by construction - it fixes a seed - but it is still Python code running on a real interpreter, so the environment checks from this topic apply unchanged. The snippet below builds the pre-flight report the fleet supervisor reads before it dispatches a mission.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()
print(f"{state['name']}: {state['battery_pct']:.1f}% "
      f"on {sys.version.split()[0]}")

## 20. Engineering Example

A reusable pre-flight function, with the policy separated from the reporting so a fleet operator can override the thresholds.

In [ ]:
REQUIRED = (3, 12)


def preflight(required=REQUIRED) -> dict:
    import sys

    problems = []
    running = tuple(sys.version_info[:2])
    if running < required:
        problems.append(f'version {running} older than {required}')
    if not sys.executable:
        problems.append('interpreter path is empty')
    return {'ready': not problems, 'problems': problems,
            'version': '.'.join(map(str, running))}


report = preflight()
print(report['version'], 'READY' if report['ready'] else report['problems'])

## 21. Guided Practice

1. Print `sys.executable`, `sys.prefix` and `sys.base_prefix`.
2. Compare `sys.prefix` with `sys.base_prefix` and state what the result tells you about the current environment.
3. Use `importlib.util.find_spec('sqlite3')` to confirm the database module is available without importing it.
4. Write `version_ok(minimum)` returning a boolean, then call it with `(3, 12)` and with `(4, 0)`.

In [ ]:
# Your turn.
import sys
import importlib.util

print('prefix      :', sys.prefix)
print('base_prefix :', sys.base_prefix)
print('sqlite3     :', importlib.util.find_spec('sqlite3') is not None)


def version_ok(minimum):
    return tuple(sys.version_info[:2]) >= tuple(minimum)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** Which check reliably identifies the interpreter actually running your code?
- **Q1.** Why is `python -m pip install X` preferable to bare `pip install X`?
- **Q1.** Which comparison correctly decides whether a machine is new enough for this course?
- **Q1.** Why should a pre-flight script return an exit status instead of only printing a verdict?

## 23. Summary

Installing Python is a three-part decision: which interpreter runs, which environment owns the packages, and which launcher the PATH selects. Every environment bug you will meet traces back to one of those three rings being different from what you assumed.

The practical habits from this topic are short and non-negotiable. Identify the interpreter with `sys.executable` instead of trusting the prompt. Install with `python -m pip` so the installer and the interpreter cannot diverge. Check optional dependencies with `importlib.util.find_spec` rather than a guarded `try: import`, because a guarded import can raise for reasons that have nothing to do with the package being present. Compare versions as tuples, never as strings. And build paths with `pathlib` so the same code works on every platform.

Above all, make the environment *observable*. A pre-flight script that reports the interpreter path, the version and the resolved module locations, then exits non-zero when something is wrong, turns an hour of guessing into a single command. That script is the first deliverable of this module, and it is exactly the kind of tool a robotics team puts on every machine.

## 24. Key Takeaways

- Three decisions: interpreter, environment, launcher.
- `sys.executable` is the authoritative answer to 'which Python am I'.
- `python -m pip` guarantees the installer matches the interpreter.
- PATH order decides which launcher wins; the first match is taken.
- `sys.path` decides which module wins an import.
- `find_spec` checks availability without executing the module.
- Compare `sys.version_info[:2]` as a tuple, never as a string.
- Return exit codes from checks so CI and shells can consume them.

## 25. Further Exploration

- [venv - Creating virtual environments](https://docs.python.org/3/library/venv.html)
- [sys - Access to the Python runtime](https://docs.python.org/3/library/sys.html)
- [importlib.util.find_spec](https://docs.python.org/3/library/importlib.html#importlib.util.find_spec)
- [PEP 405 - Virtual Environments](https://peps.python.org/pep-0405/)
- [PEP 723 - Inline script metadata](https://peps.python.org/pep-0723/) - declaring dependencies directly in a script.